In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install tifffile -q

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, Model

In [ ]:
dataset_path = "/content/drive/MyDrive/Best pannel placement/images_labels"
print("Dataset path:", dataset_path)
print("Exists:", os.path.exists(dataset_path))

**Dataset exploration**

In [ ]:
all_files = sorted(os.listdir(dataset_path))

print("Nombre total de fichiers :", len(all_files))
print("20 premiers fichiers :")
for f in all_files[:20]:
    print(f)

**Pairing images and masks**

In [ ]:
image_files = []
mask_files = []

all_tif_files = sorted([f for f in os.listdir(dataset_path) if f.endswith(".tif")])

for f in all_tif_files:
    if "_label" not in f:
        mask_name = f.replace(".tif", "_label.tif")
        if mask_name in all_tif_files:
            image_files.append(os.path.join(dataset_path, f))
            mask_files.append(os.path.join(dataset_path, mask_name))

print("Nombre d'images trouvées :", len(image_files))
print("Nombre de masques trouvés :", len(mask_files))

print("\nExemple de paire :")
print("Image :", image_files[0])
print("Mask  :", mask_files[0])

**image and mask properties**

In [ ]:
img = tiff.imread(image_files[0])
mask = tiff.imread(mask_files[0])

print("Image shape :", img.shape, "dtype:", img.dtype)
print("Mask shape  :", mask.shape, "dtype:", mask.dtype)

print("Valeurs uniques dans le masque (extrait) :")
print(np.unique(mask)[:20])

In [ ]:
def show_sample(index):
    img = tiff.imread(image_files[index])
    mask = tiff.imread(mask_files[index])

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    if img.ndim == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(img)
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='gray')
    plt.title("Mask")
    plt.axis("off")

    plt.show()

In [ ]:
for idx in [0, 1, 2]:
    show_sample(idx)

**Preprocessing**

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 8
SEED = 42

In [ ]:
def load_image(path):
    img = tiff.imread(path)

    # Si image 2D -> on la convertit en 3 canaux
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)

    # Si plus de 3 canaux, on garde seulement les 3 premiers
    if img.shape[-1] > 3:
        img = img[..., :3]

    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE)).numpy()

    # Normalisation [0,1]
    img = img.astype(np.float32)
    if img.max() > 1.0:
        img = img / 255.0

    return img


def load_mask(path):
    mask = tiff.imread(path)

    # Si masque multi-canal, on garde un seul canal
    if mask.ndim == 3:
        mask = mask[..., 0]

    mask = np.expand_dims(mask, axis=-1)
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE), method='nearest').numpy()

    # Binarisation
    mask = (mask > 0).astype(np.float32)

    return mask

**Processed sample**

In [ ]:
sample_img = load_image(image_files[0])
sample_mask = load_mask(mask_files[0])

print("Image processed shape:", sample_img.shape, sample_img.min(), sample_img.max())
print("Mask processed shape :", sample_mask.shape, np.unique(sample_mask))

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(sample_img)
plt.title("Processed image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(sample_mask.squeeze(), cmap='gray')
plt.title("Processed mask")
plt.axis("off")

plt.show()

**Data split**

In [ ]:
train_img, temp_img, train_mask, temp_mask = train_test_split(
    image_files, mask_files, test_size=0.30, random_state=SEED
)

val_img, test_img, val_mask, test_mask = train_test_split(
    temp_img, temp_mask, test_size=0.50, random_state=SEED
)

print("Train:", len(train_img))
print("Val  :", len(val_img))
print("Test :", len(test_img))

**TensorFlow pipeline**

In [ ]:
def parse_image_mask(img_path, mask_path):
    img_path = img_path.numpy().decode("utf-8")
    mask_path = mask_path.numpy().decode("utf-8")

    img = load_image(img_path)
    mask = load_mask(mask_path)

    return img, mask

def tf_parse(img_path, mask_path):
    img, mask = tf.py_function(
        parse_image_mask,
        [img_path, mask_path],
        [tf.float32, tf.float32]
    )

    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])

    return img, mask

In [ ]:
def make_dataset(img_paths, mask_paths, batch_size=8, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))
    ds = ds.map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_img, train_mask, batch_size=BATCH_SIZE, shuffle=True)
val_ds   = make_dataset(val_img, val_mask, batch_size=BATCH_SIZE, shuffle=False)
test_ds  = make_dataset(test_img, test_mask, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
for images_batch, masks_batch in train_ds.take(1):
    print("Images batch shape:", images_batch.shape)
    print("Masks batch shape :", masks_batch.shape)

**Building U-Net model**

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

def encoder_block(x, filters):
    f = conv_block(x, filters)
    p = layers.MaxPooling2D((2, 2))(f)
    return f, p

def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(256, 256, 3)):
    inputs = layers.Input(input_shape)

    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)

    b1 = conv_block(p4, 1024)

    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    outputs = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(d4)

    model = Model(inputs, outputs, name="U-Net")
    return model

**Metrics**

In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )

def iou_metric(y_true, y_pred, smooth=1e-6):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

In [ ]:
model = build_unet((IMG_SIZE, IMG_SIZE, 3))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", dice_coef, iou_metric]
)

model.summary()

**Training callbacks**

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/Best pannel placement/unet_rooftop_best.keras",
        save_best_only=True,
        monitor="val_loss"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

**Training phase**

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

**Learning curves**

In [ ]:
plt.figure(figsize=(14,4))

plt.subplot(1,3,1)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss")
plt.legend()

plt.subplot(1,3,2)
plt.plot(history.history["dice_coef"], label="train")
plt.plot(history.history["val_dice_coef"], label="val")
plt.title("Dice")
plt.legend()

plt.subplot(1,3,3)
plt.plot(history.history["iou_metric"], label="train")
plt.plot(history.history["val_iou_metric"], label="val")
plt.title("IoU")
plt.legend()

plt.show()

**Final evaluation**

In [ ]:
results = model.evaluate(test_ds)
print("Test results:", results)

**Prediction visualization**

In [ ]:
def display_prediction(dataset, model, num_samples=3):
    for images, masks in dataset.take(1):
        preds = model.predict(images)

        for i in range(num_samples):
            img = images[i].numpy()
            true_mask = masks[i].numpy().squeeze()
            pred_mask = (preds[i] > 0.5).astype(np.float32).squeeze()

            plt.figure(figsize=(12,4))

            plt.subplot(1,3,1)
            plt.imshow(img)
            plt.title("Image")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(true_mask, cmap="gray")
            plt.title("True Mask")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(pred_mask, cmap="gray")
            plt.title("Predicted Mask")
            plt.axis("off")

            plt.show()

In [ ]:
display_prediction(test_ds, model, num_samples=3)

**OVERLAY**

In [ ]:
def display_prediction_with_overlay(dataset, model, num_samples=3):
    for images, masks in dataset.take(1):
        preds = model.predict(images)

        for i in range(min(num_samples, len(images))):
            img = images[i].numpy()
            true_mask = masks[i].numpy().squeeze()
            pred_prob = preds[i].squeeze()
            pred_mask = (pred_prob > 0.5).astype(np.float32)

            plt.figure(figsize=(16,4))

            plt.subplot(1,4,1)
            plt.imshow(img)
            plt.title("Image")
            plt.axis("off")

            plt.subplot(1,4,2)
            plt.imshow(true_mask, cmap="gray")
            plt.title("True Mask")
            plt.axis("off")

            plt.subplot(1,4,3)
            plt.imshow(pred_mask, cmap="gray")
            plt.title("Predicted Mask")
            plt.axis("off")

            plt.subplot(1,4,4)
            plt.imshow(img)
            plt.imshow(pred_mask, cmap="jet", alpha=0.4)
            plt.title("Overlay")
            plt.axis("off")

            plt.show()

In [ ]:
display_prediction_with_overlay(test_ds, model, num_samples=3)

**CONFIDENCE MAP **

In [ ]:
def display_prediction_with_confidence(dataset, model, num_samples=3):
    for images, masks in dataset.take(1):
        preds = model.predict(images)

        for i in range(min(num_samples, len(images))):
            img = images[i].numpy()
            true_mask = masks[i].numpy().squeeze()
            pred_prob = preds[i].squeeze()
            pred_mask = (pred_prob > 0.5).astype(np.float32)

            plt.figure(figsize=(20,4))

            plt.subplot(1,5,1)
            plt.imshow(img)
            plt.title("Image")
            plt.axis("off")

            plt.subplot(1,5,2)
            plt.imshow(true_mask, cmap="gray")
            plt.title("True Mask")
            plt.axis("off")

            plt.subplot(1,5,3)
            plt.imshow(pred_prob, cmap="viridis")
            plt.title("Confidence Map")
            plt.axis("off")

            plt.subplot(1,5,4)
            plt.imshow(pred_mask, cmap="gray")
            plt.title("Predicted Mask")
            plt.axis("off")

            plt.subplot(1,5,5)
            plt.imshow(img)
            plt.imshow(pred_prob, cmap="jet", alpha=0.4)
            plt.title("Confidence Overlay")
            plt.axis("off")

            plt.show()

In [ ]:
display_prediction_with_confidence(test_ds, model, num_samples=3)

**FEATURE MAPS**

In [ ]:
feature_layer_names = [
    "conv2d",
    "conv2d_2",
    "conv2d_4",
    "conv2d_6"
]

In [ ]:
def show_feature_maps(model, image, layer_names, max_maps=6):
    # créer un modèle qui retourne les sorties intermédiaires
    outputs = [model.get_layer(name).output for name in layer_names]
    feature_extractor = tf.keras.models.Model(inputs=model.input, outputs=outputs)

    # ajouter batch dimension
    image_batch = np.expand_dims(image, axis=0)

    feature_maps = feature_extractor.predict(image_batch)

    for layer_name, fmap in zip(layer_names, feature_maps):
        print(f"\nLayer: {layer_name}, shape: {fmap.shape}")

        num_channels = fmap.shape[-1]
        num_show = min(max_maps, num_channels)

        plt.figure(figsize=(15, 3))
        for i in range(num_show):
            plt.subplot(1, num_show, i + 1)
            plt.imshow(fmap[0, :, :, i], cmap="viridis")
            plt.title(f"{layer_name}\nmap {i}")
            plt.axis("off")
        plt.show()

In [ ]:
for images_batch, masks_batch in test_ds.take(1):
    sample_image = images_batch[0].numpy()
    break

plt.figure(figsize=(4,4))
plt.imshow(sample_image)
plt.title("Sample Image")
plt.axis("off")
plt.show()

show_feature_maps(model, sample_image, feature_layer_names, max_maps=6)

**GRAD CAM**

In [ ]:
def make_gradcam_heatmap_segmentation(model, image, last_conv_layer_name):
    image_tensor = tf.convert_to_tensor(np.expand_dims(image, axis=0), dtype=tf.float32)

    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_tensor)

        # on prend la somme de la sortie segmentation pour avoir un score global
        loss = tf.reduce_sum(predictions)

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    heatmap = heatmap.numpy()

    return heatmap

In [ ]:
def display_gradcam(image, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    overlay = np.clip((image * 255).astype(np.uint8) * (1 - alpha) + heatmap_color * alpha, 0, 255).astype(np.uint8)

    plt.figure(figsize=(15,4))

    plt.subplot(1,3,1)
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(heatmap_resized, cmap="jet")
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(overlay)
    plt.title("Grad-CAM Overlay")
    plt.axis("off")

    plt.show()

In [ ]:
for layer in model.layers:
    print(layer.name)

In [ ]:
model.save("/content/drive/MyDrive/Best pannel placement/unet_rooftop_final.keras")

In [ ]:
# Recharger le modèle entraîné (utile après un redémarrage du kernel)
import cv2
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/Best pannel placement/unet_rooftop_final.keras",
    custom_objects={"dice_coef": dice_coef, "iou_metric": iou_metric}
)
print("✅ Modèle chargé")

In [ ]:
# === CALIBRATION VISUELLE DU GSD ===
# Sur une image, mesure la longueur d'un objet de taille connue (voiture = 4.5m)
# en pixels, puis ajuste le GSD.

import matplotlib.pyplot as plt
import tifffile as tiff

# Affiche une image avec une grille pour mesurer
img = tiff.imread(test_img[0])
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(img)
ax.set_xticks(np.arange(0, 400, 20))
ax.set_yticks(np.arange(0, 400, 20))
ax.grid(True, color='red', alpha=0.3, linewidth=0.5)
ax.set_title("Mesure une voiture (4.5m) ou une marquage routier")
plt.show()

# Compte les pixels d'une voiture sur l'image (largeur ou longueur)
CAR_LENGTH_M = 4.5
CAR_LENGTH_PX = 50   # ⚠️ AJUSTE en regardant l'image au-dessus

GSD_CALIBRATED = CAR_LENGTH_M / CAR_LENGTH_PX
print(f"GSD estimé : {GSD_CALIBRATED:.3f} m/pixel sur l'image originale 400×400")

In [ ]:
# === PARAMÈTRES DE CONVERSION SURFACE & PANNEAUX ===
GSD_SOURCE = GSD_CALIBRATED
TILE_SIZE_ORIGINAL = 400
TILE_SIZE_MODEL = IMG_SIZE  # = 256

GSD_EFFECTIVE = GSD_SOURCE * (TILE_SIZE_ORIGINAL / TILE_SIZE_MODEL)
PIXEL_AREA_M2 = GSD_EFFECTIVE ** 2

PANEL_W = 1.0
PANEL_H = 1.7
PANEL_AREA = PANEL_W * PANEL_H
PANEL_POWER_KWP = 0.4

USABLE_FRACTION = 0.70
THRESHOLD = 0.5

# === V2 — Placement géométrique ===
ROW_GAP_M = 0.10
COL_GAP_M = 0.05
EDGE_MARGIN_M = 0.30

print(f"GSD effectif sur masque 256x256 : {GSD_EFFECTIVE:.3f} m/pixel")
print(f"Surface par pixel : {PIXEL_AREA_M2:.4f} m²")
print(f"Panneau : {PANEL_W}m × {PANEL_H}m = {PANEL_AREA} m² ({PANEL_POWER_KWP} kWp)")
print(f"Fraction utilisable V1 : {USABLE_FRACTION*100:.0f}%")
print(f"Gaps V2 : row={ROW_GAP_M}m, col={COL_GAP_M}m, edge={EDGE_MARGIN_M}m")

In [ ]:
def compute_roof_metrics(pred_mask, gsd_effective=GSD_EFFECTIVE,
                          panel_area=PANEL_AREA, usable_fraction=USABLE_FRACTION,
                          panel_power=PANEL_POWER_KWP, clean=True):
    mask = (pred_mask > 0).astype(np.uint8)

    if clean:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    n_pixels_roof = int(mask.sum())
    pixel_area = gsd_effective ** 2

    roof_area_total = n_pixels_roof * pixel_area
    roof_area_usable = roof_area_total * usable_fraction

    n_panels = int(roof_area_usable // panel_area)
    estimated_kwp = round(n_panels * panel_power, 2)
    annual_production_kwh = int(estimated_kwp * 1200)

    # APRÈS — seuils calibrés sur des installations réelles
    if n_panels >= 20:
        suitability = "Excellent"
    elif n_panels >= 12:
        suitability = "Good"
    elif n_panels >= 6:
        suitability = "Limited"
    else:
        suitability = "Insufficient"

    return {
        "n_pixels_roof"        : n_pixels_roof,
        "total_roof_area_m2"   : round(roof_area_total, 1),
        "usable_roof_area_m2"  : round(roof_area_usable, 1),
        "estimated_panels"     : n_panels,
        "estimated_capacity_kwp": estimated_kwp,
        "annual_production_kwh": annual_production_kwh,
        "suitability"          : suitability,
        "cleaned_mask"         : mask,
    }

In [ ]:
def grid_pack_panels(mask, panel_h_px, panel_w_px):
    H, W = mask.shape
    if panel_h_px > H or panel_w_px > W:
        return []

    placed = []
    occupied = np.zeros_like(mask, dtype=bool)

    for y in range(0, H - panel_h_px + 1, panel_h_px):
        for x in range(0, W - panel_w_px + 1, panel_w_px):
            region_mask = mask[y:y+panel_h_px, x:x+panel_w_px]
            region_occ = occupied[y:y+panel_h_px, x:x+panel_w_px]

            if region_mask.all() and not region_occ.any():
                placed.append((y, x, panel_h_px, panel_w_px))
                occupied[y:y+panel_h_px, x:x+panel_w_px] = True

    return placed


def place_panels_geometric(mask, gsd_m_per_px=GSD_EFFECTIVE,
                            panel_w_m=PANEL_W, panel_h_m=PANEL_H,
                            row_gap_m=ROW_GAP_M, col_gap_m=COL_GAP_M,
                            edge_margin_m=EDGE_MARGIN_M):
    margin_px = max(1, int(round(edge_margin_m / gsd_m_per_px)))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                        (margin_px*2+1, margin_px*2+1))
    safe_mask = cv2.erode(mask.astype(np.uint8), kernel, iterations=1)

    pw_px = max(1, int(round((panel_w_m + col_gap_m) / gsd_m_per_px)))
    ph_px = max(1, int(round((panel_h_m + row_gap_m) / gsd_m_per_px)))

    placements_portrait  = grid_pack_panels(safe_mask, ph_px, pw_px)
    placements_landscape = grid_pack_panels(safe_mask, pw_px, ph_px)

    if len(placements_landscape) > len(placements_portrait):
        chosen, orientation = placements_landscape, "landscape"
        panel_dims_px = (pw_px, ph_px)
    else:
        chosen, orientation = placements_portrait, "portrait"
        panel_dims_px = (ph_px, pw_px)

    return {
        "placements"      : chosen,
        "n_placed"        : len(chosen),
        "orientation"     : orientation,
        "panel_dims_px"   : panel_dims_px,
        "edge_margin_px"  : margin_px,
        "safe_mask"       : safe_mask,
    }

In [ ]:
def visualize_panel_placement(image, mask, placement_result, ax=None):
    placements = placement_result["placements"]
    orientation = placement_result["orientation"]

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))

    ax.imshow(image)
    overlay = np.ma.masked_where(mask == 0, mask)
    ax.imshow(overlay, cmap="Blues", alpha=0.25)

    for (y, x, h, w) in placements:
        rect = plt.Rectangle((x, y), w, h, linewidth=0.8,
                             edgecolor="red", facecolor="#F4C430", alpha=0.75)
        ax.add_patch(rect)

    ax.set_title(f"Panel Placement: {len(placements)} panels ({orientation})")
    ax.axis("off")
    return ax

In [ ]:
def predict_rooftop_v2(image_path, model=model, threshold=THRESHOLD):
    img = load_image(image_path)
    img_batch = np.expand_dims(img, axis=0)

    pred_prob = model.predict(img_batch, verbose=0)[0, ..., 0]
    pred_mask = (pred_prob > threshold).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_CLOSE, kernel)
    cleaned_mask = cv2.morphologyEx(cleaned_mask, cv2.MORPH_OPEN, kernel)

    metrics_v1 = compute_roof_metrics(cleaned_mask, clean=False)
    placement = place_panels_geometric(cleaned_mask)

    n_panels_v2 = placement["n_placed"]
    capacity_v2_kwp = round(n_panels_v2 * PANEL_POWER_KWP, 2)
    annual_v2_kwh = int(capacity_v2_kwp * 1200)
    coverage_pct = (n_panels_v2 * PANEL_AREA / metrics_v1["total_roof_area_m2"] * 100
                    if metrics_v1["total_roof_area_m2"] > 0 else 0)

    # ✅ Suitability rating basé sur V2 (placement réel)
    if n_panels_v2 >= 20:
        suitability_v2 = "Excellent"
    elif n_panels_v2 >= 12:
        suitability_v2 = "Good"
    elif n_panels_v2 >= 6:
        suitability_v2 = "Limited"
    else:
        suitability_v2 = "Insufficient"

    return {
        "image"       : img,
        "pred_prob"   : pred_prob,
        "pred_mask"   : pred_mask,
        "cleaned_mask": cleaned_mask,
        "metrics": {
            "total_roof_area_m2"   : metrics_v1["total_roof_area_m2"],
            "usable_roof_area_m2"  : metrics_v1["usable_roof_area_m2"],
            "estimated_panels_v1"  : metrics_v1["estimated_panels"],
            "estimated_capacity_v1_kwp": metrics_v1["estimated_capacity_kwp"],
            "estimated_panels_v2"  : n_panels_v2,
            "estimated_capacity_v2_kwp": capacity_v2_kwp,
            "annual_production_v2_kwh": annual_v2_kwh,
            "panel_orientation"    : placement["orientation"],
            "real_coverage_pct"    : round(coverage_pct, 1),
            "suitability"          : suitability_v2,   # ← V2 ici
        },
        "placement"   : placement,
        "confidence"  : round(float(pred_prob[cleaned_mask > 0].mean()) if cleaned_mask.sum() > 0 else 0.0, 4),
    }

def display_full_analysis_v2(image_path, model=model):
    result = predict_rooftop_v2(image_path, model)
    m = result["metrics"]

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    axes[0].imshow(result["image"])
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(result["pred_prob"], cmap="viridis")
    axes[1].set_title(f"Confidence Map\n(mean: {result['confidence']:.2f})")
    axes[1].axis("off")

    axes[2].imshow(result["cleaned_mask"], cmap="gray")
    axes[2].set_title("Predicted Mask (cleaned)")
    axes[2].axis("off")

    visualize_panel_placement(result["image"], result["cleaned_mask"],
                              result["placement"], ax=axes[3])

    plt.tight_layout()
    plt.show()

    print("=" * 70)
    print("📊 ROOF ANALYSIS — V2 (Geometric Panel Placement)")
    print("=" * 70)
    print(f"  Total roof area              : {m['total_roof_area_m2']:>8.1f} m²")
    print(f"  Usable surface (×{int(USABLE_FRACTION*100)}%)         : {m['usable_roof_area_m2']:>8.1f} m²")
    print()
    print(f"  V1 — Theoretical estimate    : {m['estimated_panels_v1']:>4} panels  ({m['estimated_capacity_v1_kwp']:.2f} kWp)")
    print(f"  V2 — Actual placement        : {m['estimated_panels_v2']:>4} panels  ({m['estimated_capacity_v2_kwp']:.2f} kWp)  ⭐")
    print()
    print(f"  Panel orientation            : {m['panel_orientation']}")
    print(f"  Real roof coverage           : {m['real_coverage_pct']:.1f}%")
    print(f"  Estimated annual production  : {m['annual_production_v2_kwh']} kWh/year")
    print(f"  Suitability rating           : {m['suitability']}")
    print(f"  Model confidence             : {result['confidence']*100:.1f}%")
    print("=" * 70)

    return result

In [ ]:
random.seed(123)
sample_indices = random.sample(range(len(test_img)), 3)

for idx in sample_indices:
    print(f"\n🖼️  {os.path.basename(test_img[idx])}")
    result = display_full_analysis_v2(test_img[idx])

In [ ]:
import json
from pathlib import Path
import shutil

EXPORT_DIR = Path("/content/drive/MyDrive/Best pannel placement/export_rooftop")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Modèle
shutil.copy(
    "/content/drive/MyDrive/Best pannel placement/unet_rooftop_final.keras",
    EXPORT_DIR / "unet_rooftop.keras"
)

# 2. Métadonnées
meta = {
    "framework": "tensorflow",
    "tf_version": tf.__version__,
    "architecture": "U-Net (custom)",
    "task": "rooftop_segmentation + geometric_panel_placement",
    "input_shape": [256, 256, 3],
    "input_format": "RGB image, normalized 0-1",
    "output_format": "sigmoid mask 256x256x1",
    "preprocessing": {
        "resize": 256,
        "normalization": "x / 255.0",
        "color_space": "RGB"
    },
    "postprocessing": {
        "threshold": 0.5,
        "morphological_clean": True,
        "kernel_size": 5
    },
    "business_logic": {
        "dataset_source": "Kaggle - slyveinweeb/rooftop-images-for-semantic-segmentation",
        "gsd_source_m_per_px": GSD_SOURCE,
        "tile_size_original": TILE_SIZE_ORIGINAL,
        "gsd_effective_m_per_px": round(GSD_EFFECTIVE, 4),
        "panel_dimensions_m": [PANEL_W, PANEL_H],
        "panel_area_m2": PANEL_AREA,
        "panel_power_kwp": PANEL_POWER_KWP,
        "row_gap_m": ROW_GAP_M,
        "col_gap_m": COL_GAP_M,
        "edge_margin_m": EDGE_MARGIN_M,
        "usable_fraction_v1": USABLE_FRACTION,
        "annual_kwh_per_kwp": 1200
    },
    "placement_algorithm": {
        "version": "v2_geometric_grid",
        "orientations_tested": ["portrait", "landscape"],
        "selection_rule": "max_panels"
    },
    "performance": {
        "test_accuracy": 0.9788,
        "test_dice": 0.8515,
        "test_iou": 0.8292,
        "test_loss": 0.0632
    }
}
with open(EXPORT_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

# 3. Échantillons + prédictions attendues
samples_dir = EXPORT_DIR / "samples"
samples_dir.mkdir(exist_ok=True)

random.seed(42)
sample_idx = random.sample(range(len(test_img)), 5)

expected = []
for i, idx in enumerate(sample_idx):
    result = predict_rooftop_v2(test_img[idx])
    src = test_img[idx]
    dst = samples_dir / f"sample_{i+1}.tif"
    shutil.copy(src, dst)
    expected.append({
        "file": dst.name,
        "expected_total_area_m2"   : result["metrics"]["total_roof_area_m2"],
        "expected_panels_v1"       : result["metrics"]["estimated_panels_v1"],
        "expected_panels_v2"       : result["metrics"]["estimated_panels_v2"],
        "expected_capacity_v2_kwp" : result["metrics"]["estimated_capacity_v2_kwp"],
        "expected_orientation"     : result["metrics"]["panel_orientation"],
    })

with open(samples_dir / "expected.json", "w") as f:
    json.dump(expected, f, indent=2)

# APRÈS (corrigé)
print("✅ Export V2 terminé :")
for f in EXPORT_DIR.rglob("*"):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        rel = str(f.relative_to(EXPORT_DIR))   # ← conversion en str
        print(f"   {rel:<40} ({size_kb:>8.1f} KB)")